# YOLO Dataset Preparation & Training

This notebook combines dataset preparation and training for YOLO parking space detection.

**Part 1: Dataset Preparation**
- Convert HuggingFace XML annotations to YOLO format
- Create train/val splits
- Generate data.yaml configuration

**Part 2: Model Training**
- Train YOLOv8 model
- Monitor training metrics
- Run inference on images and videos


## 1. Imports and Path Setup


In [ ]:
from pathlib import Path
import xml.etree.ElementTree as ET
import shutil
from sklearn.model_selection import train_test_split
from tqdm import tqdm

BASE_DIR = Path.cwd().parent.parent if Path.cwd().name == "Phase 1" else (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())

# HuggingFace dataset paths
# Note: annotations.xml is at hf_parking_space/annotations.xml (not inside data/)
HF_ROOT = BASE_DIR / "data" / "raw" / "hf_parking_space"
HF_IMAGES_DIR = HF_ROOT / "data" / "images"
HF_ANNOT_XML = HF_ROOT / "annotations.xml"  # XML is at root level, not in data/

# YOLO dataset output paths
YOLO_ROOT = BASE_DIR / "data" / "processed" / "yolo_parking"
IMG_TRAIN = YOLO_ROOT / "images" / "train"
IMG_VAL = YOLO_ROOT / "images" / "val"
LBL_TRAIN = YOLO_ROOT / "labels" / "train"
LBL_VAL = YOLO_ROOT / "labels" / "val"

# Create directories
for d in [IMG_TRAIN, IMG_VAL, LBL_TRAIN, LBL_VAL]:
    d.mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("HF_ROOT:", HF_ROOT)
print("HF_IMAGES_DIR:", HF_IMAGES_DIR, "exists:", HF_IMAGES_DIR.exists())
print("HF_ANNOT_XML:", HF_ANNOT_XML, "exists:", HF_ANNOT_XML.exists())
print("YOLO_ROOT:", YOLO_ROOT)


BASE_DIR: c:\Harosha\George Brown\DL2\parking-vision
HF_ROOT: c:\Harosha\George Brown\DL2\parking-vision\data\raw\hf_parking_space
HF_IMAGES_DIR: c:\Harosha\George Brown\DL2\parking-vision\data\raw\hf_parking_space\data\images exists: True
HF_ANNOT_XML: c:\Harosha\George Brown\DL2\parking-vision\data\raw\hf_parking_space\annotations.xml exists: True
YOLO_ROOT: c:\Harosha\George Brown\DL2\parking-vision\data\processed\yolo_parking


## 2. Inspect XML Structure

In [2]:
# Inspect the XML structure to understand the format
if HF_ANNOT_XML.exists():
    tree = ET.parse(HF_ANNOT_XML)
    root = tree.getroot()
    
    print("Root tag:", root.tag)
    print("\nFirst 3 image elements:")
    for i, child in enumerate(list(root)[:3]):
        print(f"\n--- Image {i+1} ---")
        ET.dump(child)
        if i >= 2:
            break
else:


Root tag: annotations

First 3 image elements:

--- Image 1 ---
<version>1.1</version>
  

--- Image 2 ---
<meta>
    <task>
      <segments>
        <segment>
          <id>32599</id>
          <start>0</start>
          <stop>32</stop>
          <url>https://cvat2.trainingdata.solutions/api/jobs/32599</url>
        </segment>
      </segments>
      <owner>
        <username>TrainingData</username>
        <email />
      </owner>
      
      <labels>
        <label>
          <name>free_parking_space</name>
          <color>#3d3df5</color>
          <type>polygon</type>
          <attributes>
            <attribute>
              <name>not_visible</name>
              <mutable>False</mutable>
              <input_type>checkbox</input_type>
              <default_value>false</default_value>
              <values>false</values>
            </attribute>
          </attributes>
        </label>
        <label>
          <name>not_free_parking_space</name>
          <color>#ff6037</colo

## 3. Parse XML Annotations


In [3]:
def parse_hf_xml(xml_path: Path):
    """
    Parse HuggingFace XML annotations (CVAT format).
    Returns a dictionary mapping image names to their annotations.
    """
    tree = ET.parse(xml_path)
    root = tree.getroot()

    images_info = {}  # {img_name: {"width": w, "height": h, "boxes": [dicts...] } }

    for img_elem in root.findall("image"):
        img_name = img_elem.get("name")
        img_w = float(img_elem.get("width"))
        img_h = float(img_elem.get("height"))

        boxes = []
        
        # Try different possible tag names for bounding boxes
        # CVAT uses "box", but some formats use "polygon" which we convert to bbox
        for box in img_elem.findall("box"):
            label = box.get("label")
            xtl = float(box.get("xtl"))
            ytl = float(box.get("ytl"))
            xbr = float(box.get("xbr"))
            ybr = float(box.get("ybr"))

            boxes.append({
                "label": label,
                "xtl": xtl,
                "ytl": ytl,
                "xbr": xbr,
                "ybr": ybr,
            })
        
        # Also handle polygon annotations (convert to bounding box)
        for polygon in img_elem.findall("polygon"):
            label = polygon.get("label")
            points_str = polygon.get("points")
            
            # Parse points: "x1,y1;x2,y2;x3,y3;x4,y4"
            points = []
            for point_pair in points_str.split(";"):
                if point_pair.strip():
                    x, y = map(float, point_pair.split(","))
                    points.append((x, y))
            
            if len(points) < 2:
                continue
            
            # Convert polygon to bounding box
            x_coords = [p[0] for p in points]
            y_coords = [p[1] for p in points]
            xtl = min(x_coords)
            ytl = min(y_coords)
            xbr = max(x_coords)
            ybr = max(y_coords)
            
            boxes.append({
                "label": label,
                "xtl": xtl,
                "ytl": ytl,
                "xbr": xbr,
                "ybr": ybr,
            })

        if boxes:  # Only add images that have annotations
            images_info[img_name] = {
                "width": img_w,
                "height": img_h,
                "boxes": boxes,
            }

    return images_info

# Parse the XML
images_info = parse_hf_xml(HF_ANNOT_XML)
print(f"Parsed {len(images_info)} images with annotations")
print(f"\nSample image names:")
for i, img_name in enumerate(list(images_info.keys())[:5]):
    img_info = images_info[img_name]
    print(f"  {i+1}. {img_name}: {len(img_info['boxes'])} boxes, size: {img_info['width']}x{img_info['height']}")


Parsed 30 images with annotations

Sample image names:
  1. images/0.png: 28 boxes, size: 1200.0x621.0
  2. images/1.png: 38 boxes, size: 650.0x487.0
  3. images/10.png: 20 boxes, size: 2560.0x1820.0
  4. images/11.png: 36 boxes, size: 1353.0x1041.0
  5. images/12.png: 39 boxes, size: 1920.0x1080.0


In [4]:
# Map labels to YOLO class IDs
label_to_id = {
    "free_parking_space": 0,
    "not_free_parking_space": 1,
    "partially_free_parking_space": 2,
}

id_to_label = {v: k for k, v in label_to_id.items()}

print("Label to ID mapping:")
for label, cls_id in label_to_id.items():
    print(f"  {cls_id}: {label}")

# Check which labels appear in the dataset
all_labels = set()
for img_info in images_info.values():
    for box in img_info["boxes"]:
        all_labels.add(box["label"])

print(f"\nLabels found in dataset: {sorted(all_labels)}")
print(f"Labels in mapping: {set(label_to_id.keys())}")

missing_labels = all_labels - set(label_to_id.keys())
if missing_labels:
    print(f"\nWARNING: Some labels are not in the mapping: {missing_labels}")
    print("These will be skipped during conversion.")


Label to ID mapping:
  0: free_parking_space
  1: not_free_parking_space
  2: partially_free_parking_space

Labels found in dataset: ['free_parking_space', 'not_free_parking_space', 'partially_free_parking_space']
Labels in mapping: {'not_free_parking_space', 'partially_free_parking_space', 'free_parking_space'}


## 4. Train/Validation Split


In [5]:
# Split images into train and validation sets
all_img_names = list(images_info.keys())
train_imgs, val_imgs = train_test_split(
    all_img_names,
    test_size=0.2,
    random_state=42,
    shuffle=True,
)

print(f"Total images: {len(all_img_names)}")
print(f"Train images: {len(train_imgs)} ({len(train_imgs)/len(all_img_names)*100:.1f}%)")
print(f"Val images: {len(val_imgs)} ({len(val_imgs)/len(all_img_names)*100:.1f}%)")


Total images: 30
Train images: 24 (80.0%)
Val images: 6 (20.0%)


## 5. Convert to YOLO Format


def boxes_to_yolo_lines(img_info, label_map):
    """
    Convert image info boxes to YOLO format label lines.
    
    Args:
        img_info: dict with 'width', 'height', 'boxes' keys
        label_map: dict mapping label strings to class IDs
    
    Returns:
        List of strings, each string is a YOLO format line: "class_id x_center y_center width height"
    """
    lines = []
    img_w = img_info["width"]
    img_h = img_info["height"]
    
    for box in img_info["boxes"]:
        label = box["label"]
        if label not in label_map:
            continue
        
        cls_id = label_map[label]
        
        # Convert from (xtl, ytl, xbr, ybr) to YOLO format (x_center, y_center, width, height) normalized
        xtl = box["xtl"]
        ytl = box["ytl"]
        xbr = box["xbr"]
        ybr = box["ybr"]
        
        # Ensure valid box
        if xbr <= xtl or ybr <= ytl:
            continue
        
        # Calculate center and dimensions in pixel coordinates
        x_center = (xtl + xbr) / 2.0
        y_center = (ytl + ybr) / 2.0
        width = xbr - xtl
        height = ybr - ytl
        
        # Normalize to [0, 1]
        x_center_norm = x_center / img_w
        y_center_norm = y_center / img_h
        width_norm = width / img_w
        height_norm = height / img_h
        
        # Clamp to valid range
        x_center_norm = max(0.0, min(1.0, x_center_norm))
        y_center_norm = max(0.0, min(1.0, y_center_norm))
        width_norm = max(0.0, min(1.0, width_norm))
        height_norm = max(0.0, min(1.0, height_norm))
        
        # Format: "class_id x_center y_center width height"
        line = f"{cls_id} {x_center_norm:.6f} {y_center_norm:.6f} {width_norm:.6f} {height_norm:.6f}"
        lines.append(line)
    
    return lines

print("boxes_to_yolo_lines function defined")


In [7]:
def process_split(img_names, img_dst_dir, lbl_dst_dir, images_info, label_map, split_name="train"):
    """
    Process a split (train or val): copy images and create YOLO label files.
    """
    count = 0
    skipped_no_image = 0
    skipped_no_labels = 0
    
    for img_name in tqdm(img_names, desc=f"Processing {split_name}"):
        if img_name not in images_info:
            continue

        img_info = images_info[img_name]

        # Source image path
        # Handle "images/0.png" format - extract just the filename
        img_filename = Path(img_name).name
        src_img_path = HF_IMAGES_DIR / img_filename
        
        if not src_img_path.exists():
            skipped_no_image += 1
            continue

        # Compute YOLO label lines
        lines = boxes_to_yolo_lines(img_info, label_map)
        if not lines:
            skipped_no_labels += 1
            continue

        # Copy image
        dst_img_path = img_dst_dir / img_filename
        shutil.copy2(src_img_path, dst_img_path)

        # Write label file (same name as image but with .txt extension)
        dst_lbl_path = lbl_dst_dir / (Path(img_filename).stem + ".txt")
        with open(dst_lbl_path, "w") as f:
            f.write("\n".join(lines))

        count += 1

    return count, skipped_no_image, skipped_no_labels

# Process train and validation splits
print("Processing train split...")
n_train, skipped_train_img, skipped_train_lbl = process_split(
    train_imgs, IMG_TRAIN, LBL_TRAIN, images_info, label_to_id, "train"
)

print("\nProcessing validation split...")
n_val, skipped_val_img, skipped_val_lbl = process_split(
    val_imgs, IMG_VAL, LBL_VAL, images_info, label_to_id, "val"
)

print(f"\n{'='*60}")
print(f"Conversion Summary:")
print(f"{'='*60}")
print(f"Train images converted: {n_train}")
print(f"  - Skipped (no image file): {skipped_train_img}")
print(f"  - Skipped (no valid labels): {skipped_train_lbl}")
print(f"\nVal images converted: {n_val}")
print(f"  - Skipped (no image file): {skipped_val_img}")
print(f"  - Skipped (no valid labels): {skipped_val_lbl}")
print(f"\nTotal: {n_train + n_val} images converted")


Processing train split...


Processing train: 100%|██████████| 24/24 [00:00<00:00, 59.99it/s]



Processing validation split...


Processing val: 100%|██████████| 6/6 [00:00<00:00, 63.60it/s]


Conversion Summary:
Train images converted: 24
  - Skipped (no image file): 0
  - Skipped (no valid labels): 0

Val images converted: 6
  - Skipped (no image file): 0
  - Skipped (no valid labels): 0

Total: 30 images converted


In [8]:
# Create data.yaml for YOLO training
data_yaml = f"""path: {YOLO_ROOT.as_posix()}

train: images/train
val: images/val

names:
  0: free_parking_space
  1: not_free_parking_space
  2: partially_free_parking_space
"""

data_yaml_path = YOLO_ROOT / "data.yaml"
with open(data_yaml_path, "w") as f:
    f.write(data_yaml)

print("Created data.yaml:")
print("="*60)
print(data_yaml_path.read_text())
print("="*60)
print(f"\nFile saved to: {data_yaml_path}")


Created data.yaml:
path: c:/Harosha/George Brown/DL2/parking-vision/data/processed/yolo_parking

train: images/train
val: images/val

names:
  0: free_parking_space
  1: not_free_parking_space
  2: partially_free_parking_space


File saved to: c:\Harosha\George Brown\DL2\parking-vision\data\processed\yolo_parking\data.yaml


## 6. Verify Dataset Structure


In [9]:
# Verify the dataset structure
print("Dataset structure verification:")
print("="*60)

# Count files
train_imgs_count = len(list(IMG_TRAIN.glob("*")))
train_lbls_count = len(list(LBL_TRAIN.glob("*.txt")))
val_imgs_count = len(list(IMG_VAL.glob("*")))
val_lbls_count = len(list(LBL_VAL.glob("*.txt")))

print(f"Train images: {train_imgs_count}")
print(f"Train labels: {train_lbls_count}")
print(f"Val images: {val_imgs_count}")
print(f"Val labels: {val_lbls_count}")

if train_imgs_count == train_lbls_count and val_imgs_count == val_lbls_count:
    print("\n✓ Dataset structure is correct (image count matches label count)")
else:
    print("\n⚠ WARNING: Image and label counts don't match!")

# Check a sample label file
if list(LBL_TRAIN.glob("*.txt")):
    sample_label = list(LBL_TRAIN.glob("*.txt"))[0]
    print(f"\nSample label file ({sample_label.name}):")
    print(sample_label.read_text()[:200])



Dataset structure verification:
Train images: 24
Train labels: 24
Val images: 6
Val labels: 6

✓ Dataset structure is correct (image count matches label count)

Sample label file (0.txt):
0 0.081633 0.171739 0.071933 0.319002
0 0.022462 0.823816 0.044925 0.311079
0 0.023262 0.169944 0.046525 0.318535
0 0.961463 0.162005 0.073192 0.315572
0 0.888008 0.162585 0.073200 0.315572
0 0.738712

Dataset ready at: c:\Harosha\George Brown\DL2\parking-vision\data\processed\yolo_parking
Next step: Train YOLO model using notebook 05_yolo_train.ipynb


---

## Part 2: Model Training

After preparing the dataset above, proceed with training the YOLO model.


## 7. Setup and Configuration


In [ ]:
from ultralytics import YOLO
from pathlib import Path
import torch

BASE_DIR = Path.cwd().parent.parent if Path.cwd().name == "Phase 1" else (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
YOLO_ROOT = BASE_DIR / "data" / "processed" / "yolo_parking"
DATA_YAML = YOLO_ROOT / "data.yaml"
RUNS_DIR = BASE_DIR / "runs"

print("BASE_DIR:", BASE_DIR)
print("DATA_YAML:", DATA_YAML, "exists:", DATA_YAML.exists())
print("YOLO_ROOT:", YOLO_ROOT, "exists:", YOLO_ROOT.exists())

# Check device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

if DATA_YAML.exists():
    print("\ndata.yaml contents:")
    print("="*60)
    print(DATA_YAML.read_text())
else:


BASE_DIR: c:\Harosha\George Brown\DL2\parking-vision
DATA_YAML: c:\Harosha\George Brown\DL2\parking-vision\data\processed\yolo_parking\data.yaml exists: True
YOLO_ROOT: c:\Harosha\George Brown\DL2\parking-vision\data\processed\yolo_parking exists: True
Using device: cpu

data.yaml contents:
path: c:/Harosha/George Brown/DL2/parking-vision/data/processed/yolo_parking

train: images/train
val: images/val

names:
  0: free_parking_space
  1: not_free_parking_space
  2: partially_free_parking_space



## 8. Load Pretrained Model


In [ ]:
# Choose model size: yolov8n (nano), yolov8s (small), yolov8m (medium), yolov8l (large), yolov8x (xlarge)
# For faster training and inference, start with yolov8n
# For better accuracy, use yolov8s or yolov8m

MODEL_SIZE = "n"  # Options: "n", "s", "m", "l", "x"
MODEL_NAME = f"yolov8{MODEL_SIZE}.pt"

print(f"Loading pretrained model: {MODEL_NAME}")
model = YOLO(MODEL_NAME)

print(f"Model loaded: {MODEL_NAME}")
print(f"Model will be fine-tuned on parking space detection dataset")


Loading pretrained model: yolov8n.pt
Model loaded: yolov8n.pt
Model will be fine-tuned on parking space detection dataset


## 9. Training Configuration


In [ ]:
# Training hyperparameters
TRAIN_CONFIG = {
    "data": str(DATA_YAML),
    "epochs": 50,              # Number of training epochs
    "imgsz": 640,              # Image size (640 is standard for YOLO)
    "batch": 16,               # Batch size (adjust based on GPU memory)
    "name": "yolo_parking_v1", # Experiment name
    "project": str(RUNS_DIR),  # Project directory
    "patience": 10,            # Early stopping patience
    "save": True,              # Save checkpoints
    "save_period": 10,         # Save checkpoint every N epochs
    "val": True,               # Validate during training
    "plots": True,             # Generate training plots
    "device": device,          # Device to use
}

print("Training configuration:")
print("="*60)
for key, value in TRAIN_CONFIG.items():
    print(f"  {key}: {value}")
print("="*60)


Training configuration:
  data: c:\Harosha\George Brown\DL2\parking-vision\data\processed\yolo_parking\data.yaml
  epochs: 50
  imgsz: 640
  batch: 16
  name: yolo_parking_v1
  project: c:\Harosha\George Brown\DL2\parking-vision\runs
  patience: 10
  save: True
  save_period: 10
  val: True
  plots: True
  device: cpu


## 10. Train Model


In [4]:


results = model.train(**TRAIN_CONFIG)


Ultralytics 8.3.228  Python-3.12.4 torch-2.5.1+cu121 CPU (AMD Ryzen 7 7730U with Radeon Graphics)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=c:\Harosha\George Brown\DL2\parking-vision\data\processed\yolo_parking\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolo_parking_v1, nbs=64, nms=False, opset=None, optimize=False, optimize

## 11. Training Results Summary


In [ ]:
# Training results are saved in runs/detect/yolo_parking_v1/
results_dir = RUNS_DIR / "detect" / TRAIN_CONFIG["name"]
best_model_path = results_dir / "weights" / "best.pt"
last_model_path = results_dir / "weights" / "last.pt"

print("Training results:")
print("="*60)
print(f"Results directory: {results_dir}")
print(f"Best model: {best_model_path}")
print(f"Last model: {last_model_path}")

if best_model_path.exists():
    print(f"\n✓ Best model saved: {best_model_path}")
    file_size_mb = best_model_path.stat().st_size / (1024 * 1024)
    print(f"  File size: {file_size_mb:.2f} MB")
else:
    print("\n⚠ Best model not found!")

# Display key metrics if available
if hasattr(results, 'results_dict'):
    print("\nFinal metrics:")
    for key, value in results.results_dict.items():
        if isinstance(value, (int, float)):
            print(f"  {key}: {value:.4f}")


<>:28: SyntaxWarning: invalid escape sequence '\ '
<>:28: SyntaxWarning: invalid escape sequence '\ '


C:\Users\haros\AppData\Local\Temp\ipykernel_7560\3206923458.py:28: SyntaxWarning: invalid escape sequence '\ '
  print("\ Best model not found!")


Training results:
Results directory: c:\Harosha\George Brown\DL2\parking-vision\runs\yolo_parking_v1
Best model: c:\Harosha\George Brown\DL2\parking-vision\runs\yolo_parking_v1\weights\best.pt
Last model: c:\Harosha\George Brown\DL2\parking-vision\runs\yolo_parking_v1\weights\last.pt

 Best model saved: c:\Harosha\George Brown\DL2\parking-vision\runs\yolo_parking_v1\weights\best.pt
  File size: 5.96 MB

 Final metrics:
  metrics/precision(B): 0.9050
  metrics/recall(B): 0.4706
  metrics/mAP50(B): 0.5429
  metrics/mAP50-95(B): 0.4320
  fitness: 0.4320


## 12. Load Best Model for Inference


In [ ]:
# Load the best model
if best_model_path.exists():
    best_model = YOLO(str(best_model_path))
    print(f"Loaded best model from: {best_model_path}")
    print("Model ready for inference!")
else:
    print(f"ERROR: Best model not found at {best_model_path}")
    best_model = None


Loaded best model from: c:\Harosha\George Brown\DL2\parking-vision\runs\yolo_parking_v1\weights\best.pt


## 13. Inference on Validation Images


In [ ]:
if best_model is not None:
    # Get validation images
    from glob import glob
    
    val_images_dir = YOLO_ROOT / "images" / "val"
    val_images = glob(str(val_images_dir / "*.jpg")) + glob(str(val_images_dir / "*.png"))
    
        test_images = val_images[:num_test_images]
        
        print(f"\nRunning inference on {num_test_images} validation images...")
        
        results = best_model.predict(
            source=test_images,
            save=True,
            project=str(RUNS_DIR),
            name="yolo_parking_pred_images",
            conf=0.25,        # Confidence threshold
            imgsz=640,       # Image size
            show_labels=True,
            show_conf=True,
        )
        
        print(f"\n✓ Inference complete!")
        print(f"Annotated images saved to: {RUNS_DIR / 'detect' / 'yolo_parking_pred_images'}")
        
        # Display results summary
        if results:
            print(f"\nProcessed {len(results)} images")
            for i, result in enumerate(results[:3]):  # Show first 3
                print(f"  Image {i+1}: {len(result.boxes)} detections")
    else:
        print("No validation images found!")
else:


Found 6 validation images

Running inference on 5 validation images...

0: 640x640 11 not_free_parking_spaces, 123.5ms
1: 640x640 1 free_parking_space, 25 not_free_parking_spaces, 123.5ms
2: 640x640 6 free_parking_spaces, 21 not_free_parking_spaces, 123.5ms
3: 640x640 27 free_parking_spaces, 36 not_free_parking_spaces, 123.5ms
4: 640x640 13 free_parking_spaces, 11 not_free_parking_spaces, 123.5ms
Speed: 12.3ms preprocess, 123.5ms inference, 9.7ms postprocess per image at shape (1, 3, 640, 640)
Results saved to C:\Harosha\George Brown\DL2\parking-vision\runs\yolo_parking_pred_images

 Inference complete!
Annotated images saved to: c:\Harosha\George Brown\DL2\parking-vision\runs\detect\yolo_parking_pred_images

Processed 5 images
  Image 1: 11 detections
  Image 2: 26 detections
  Image 3: 27 detections


## 14. Inference on Video


In [ ]:
if best_model is not None:
    # Video paths
    VIDEO_DIR = BASE_DIR / "data" / "raw" / "UFPARK - Dataset" / "morning_samples_anonimized"
    video_files = list(VIDEO_DIR.glob("*.avi")) + list(VIDEO_DIR.glob("*.mp4"))
    
    if video_files:
        # Select first video (or change index)
        SELECTED_VIDEO_INDEX = 0
        VIDEO_INPUT = video_files[SELECTED_VIDEO_INDEX]
        
        print(f"Selected video: {VIDEO_INPUT.name}")
        print(f"Running YOLO inference on video...")
        print("This may take a while depending on video length.")
        print("="*60)
        
        results = best_model.predict(
            source=str(VIDEO_INPUT),
            save=True,
            project=str(RUNS_DIR),
            name="yolo_parking_video_v1",
            conf=0.25,        # Confidence threshold
            imgsz=640,       # Image size
            show_labels=True,
            show_conf=True,
        )
        
        # Output video path
        output_video_dir = RUNS_DIR / "detect" / "yolo_parking_video_v1"
        output_video = output_video_dir / VIDEO_INPUT.name
        
        print(f"\n✓ Video inference complete!")
        print(f"Annotated video saved to: {output_video}")
        
        if output_video.exists():
            file_size_mb = output_video.stat().st_size / (1024 * 1024)
            print(f"Output file size: {file_size_mb:.2f} MB")
    else:
        print(f"No video files found in {VIDEO_DIR}")
else:


Selected video: 2020-03-19-08-00-00.scene_0000TESTE_BB_write.avi

WARNING 
inference results will accumulate in RAM unless `stream=True` is passed, causing potential out-of-memory
errors for large sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/599) c:\Harosha\George Brown\DL2\parking-vision\data\raw\UFPARK - Dataset\morning_samples_anonimized\2020-03-19-08-00-00.scene_0000TESTE_BB_write.avi: 480x640 6 free_parking_spaces, 6 not_free_parking_spaces, 145.9ms
video 1/1 (frame 2/599) c:\Harosha\George Brown\DL2\parking-vision\data\raw\UFPARK - Dataset\morning_samples_anonimized\2020-03-19-08-00-00.scene_0000TESTE_BB

In [ ]:
# Set to True to process all videos
PROCESS_ALL_VIDEOS = False

if PROCESS_ALL_VIDEOS and best_model is not None:
    VIDEO_DIR = BASE_DIR / "data" / "raw" / "UFPARK - Dataset" / "morning_samples_anonimized"
    video_files = list(VIDEO_DIR.glob("*.avi")) + list(VIDEO_DIR.glob("*.mp4"))
    video_files.sort()
    
    print(f"Processing {len(video_files)} videos...")
    print("="*60)
    
    for i, video_path in enumerate(video_files):
        print(f"\n[{i+1}/{len(video_files)}] Processing: {video_path.name}")
        
        results = best_model.predict(
            source=str(video_path),
            save=True,
            project=str(RUNS_DIR),
            name=f"yolo_parking_video_batch",
            conf=0.25,
            imgsz=640,
            show_labels=True,
            show_conf=True,
        )
        
        output_video_dir = RUNS_DIR / "detect" / "yolo_parking_video_batch"
        output_video = output_video_dir / video_path.name
        
        if output_video.exists():
            file_size_mb = output_video.stat().st_size / (1024 * 1024)
            print(f"  ✓ Saved: {output_video.name} ({file_size_mb:.2f} MB)")
    
    print(f"\n{'='*60}")
    print(f"Batch processing complete! Processed {len(video_files)} videos.")
    print(f"Output directory: {RUNS_DIR / 'detect' / 'yolo_parking_video_batch'}")
else:
    if not PROCESS_ALL_VIDEOS:
    elif best_model is None:


## 15. Summary

The YOLO object detection pipeline is now complete:

1. **Dataset Preparation**: Converted HuggingFace XML annotations to YOLO format
2. **Model Training**: Trained YOLOv8 model on parking space detection
3. **Inference**: Tested on validation images and videos

**Key Files:**
- Best model: `runs/detect/yolo_parking_v1/weights/best.pt`
- Training plots: `runs/detect/yolo_parking_v1/results.png`
- Annotated images: `runs/detect/yolo_parking_pred_images/`
- Annotated videos: `runs/detect/yolo_parking_video_v1/`

**Next Steps:**
- Evaluate model performance on test set
- Fine-tune hyperparameters if needed
- Compare with CNN baseline results
- Deploy model for real-time inference
